# NSW EV Intelligence Platform - RAG Vector Search Setup

This notebook sets up vector search for a Retrieval-Augmented Generation (RAG) application on your NSW EV data.

## What This Does
1. **Prepares documents** - Converts structured EV data into text documents suitable for semantic search
2. **Generates embeddings** - Uses Databricks Foundation Models to create vector embeddings
3. **Creates vector search endpoint** - Sets up compute for hosting the search index
4. **Creates search index** - Builds a Delta Sync index with automatic updates
5. **Tests retrieval** - Validates the search works correctly

## Data Sources
- `mobility_ai.silver.ev_charging_stations` - 1,950+ charging stations
- `mobility_ai.gold.trip_routes_optimal` - Route safety and hazard analysis
- `mobility_ai.silver.fuel_prices` - Fuel pricing for transition analysis

In [0]:
%pip install databricks-vectorsearch databricks-sdk --upgrade --quiet
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType, 
    DeltaSyncVectorIndexSpecRequest, 
    EmbeddingSourceColumn, 
    PipelineType,
    VectorIndexType
)
from databricks.vector_search.client import VectorSearchClient
import time

# Initialize clients
w = WorkspaceClient()
vsc = VectorSearchClient()

# Configuration
CATALOG = "mobility_ai"
SCHEMA_RAG = "rag"
DOCUMENTS_TABLE = f"{CATALOG}.{SCHEMA_RAG}.ev_documents"
INDEX_NAME = f"{CATALOG}.{SCHEMA_RAG}.ev_documents_index"
ENDPOINT_NAME = "nsw-ev-rag-endpoint"

print(f"Catalog: {CATALOG}")
print(f"RAG Schema: {SCHEMA_RAG}")
print(f"Documents Table: {DOCUMENTS_TABLE}")
print(f"Index Name: {INDEX_NAME}")
print(f"Endpoint Name: {ENDPOINT_NAME}")

## Step 1: Create RAG Schema

Create a dedicated schema in Unity Catalog to store RAG-related tables (documents, embeddings, indexes).

In [0]:
%sql
-- Create RAG schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS mobility_ai.rag
COMMENT 'Schema for RAG application documents and vector search indexes';

## Step 2: Prepare Documents Table

Convert structured EV data into text documents suitable for semantic search. Each document will contain:
- **EV Charging Stations**: Station details, operator, location, charging specs
- **Route Safety**: Hazard summaries, risk scores, safety ratings

The documents table requires:
- `doc_id` (STRING) - Primary key for the document
- `content` (STRING) - Text content to be embedded and searched
- `metadata` (STRING) - JSON metadata about the document source
- `source_table` (STRING) - Which source table this came from

In [0]:
%sql
-- Create documents table from EV charging stations
CREATE OR REPLACE TABLE mobility_ai.rag.ev_documents (
  doc_id STRING NOT NULL,
  content STRING NOT NULL,
  metadata STRING,
  source_table STRING,
  CONSTRAINT pk_doc_id PRIMARY KEY (doc_id)
)
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
)
COMMENT 'RAG documents for NSW EV Intelligence Platform';

-- Insert EV charging station documents
INSERT INTO mobility_ai.rag.ev_documents
SELECT 
  CONCAT('ev_station_', CAST(objectid AS STRING)) AS doc_id,
  CONCAT(
    'Electric Vehicle Charging Station: ', COALESCE(station_name, 'Unnamed Station'), '. ',
    'Location: ', COALESCE(station_address, 'Address not available'), ', ', 
    COALESCE(lganame, 'Unknown LGA'), ' (Postcode: ', COALESCE(CAST(pcode AS STRING), 'N/A'), '). ',
    'Operator: ', COALESCE(operator, 'Unknown operator'), '. ',
    'Charging Type: ', COALESCE(charger__type, 'Not specified'), ' with ', 
    COALESCE(CAST(number_of_plugs AS STRING), '0'), ' plugs. ',
    'Charging Speed: ', COALESCE(charging_speed, 'Unknown'), ' (', 
    COALESCE(CAST(charger_rating_kw AS STRING), 'N/A'), ' kW). ',
    'Coordinates: Latitude ', CAST(latitude AS STRING), ', Longitude ', CAST(longitude AS STRING), '.'
  ) AS content,
  TO_JSON(STRUCT(
    objectid,
    station_name,
    operator,
    lganame,
    pcode,
    latitude,
    longitude,
    charger_rating_kw,
    charging_speed
  )) AS metadata,
  'mobility_ai.silver.ev_charging_stations' AS source_table
FROM mobility_ai.silver.ev_charging_stations
WHERE has_valid_location = true
  AND objectid IS NOT NULL;

In [0]:
%sql
-- Insert route safety and hazard documents
INSERT INTO mobility_ai.rag.ev_documents
SELECT 
  CONCAT('route_', REPLACE(location, ' ', '_')) AS doc_id,
  CONCAT(
    'Route Safety Report for ', location, '. ',
    'Overall Safety Rating: ', route_safety_rating, ' with a risk score of ', 
    CAST(ROUND(route_risk_score, 2) AS STRING), '. ',
    'Total Hazards: ', CAST(total_hazards AS STRING), ' (', 
    CAST(active_hazards AS STRING), ' currently active). ',
    'Hazard Breakdown: ', CAST(roadwork_count AS STRING), ' roadwork sites, ',
    CAST(incident_count AS STRING), ' incidents, ',
    CAST(flood_count AS STRING), ' flood warnings, ',
    CAST(event_count AS STRING), ' events. ',
    'Primary Hazard Type: ', COALESCE(primary_hazard_type, 'None'), '. ',
    'Estimated Delay: ', CAST(estimated_delay_minutes AS STRING), ' minutes. ',
    'Location Center: Latitude ', CAST(center_lat AS STRING), 
    ', Longitude ', CAST(center_lon AS STRING), '.'
  ) AS content,
  TO_JSON(STRUCT(
    location,
    route_safety_rating,
    route_risk_score,
    total_hazards,
    active_hazards,
    primary_hazard_type,
    estimated_delay_minutes,
    center_lat,
    center_lon
  )) AS metadata,
  'mobility_ai.gold.trip_routes_optimal' AS source_table
FROM mobility_ai.gold.trip_routes_optimal
WHERE total_hazards > 0;

In [0]:
%sql
-- Check documents created
SELECT 
  source_table,
  COUNT(*) as document_count,
  AVG(LENGTH(content)) as avg_content_length
FROM mobility_ai.rag.ev_documents
GROUP BY source_table
ORDER BY document_count DESC;

In [0]:
%sql
-- View sample documents
SELECT 
  doc_id,
  SUBSTRING(content, 1, 200) as content_preview,
  source_table
FROM mobility_ai.rag.ev_documents
LIMIT 5;

## Step 3: Create Vector Search Endpoint

Vector Search endpoints provide compute resources for hosting indexes. We'll use a **Standard** endpoint for low-latency retrieval (20-50ms response time).

**Endpoint Types:**
- **Standard**: Low latency (20-50ms), up to 320M vectors, best for real-time applications
- **Storage-Optimized**: Higher latency (300-500ms), 1B+ vectors, 7x lower cost

For this RAG application, Standard is recommended for responsive chat experiences.

In [0]:
# Check if endpoint exists, create if not
try:
    endpoint = w.vector_search_endpoints.get_endpoint(ENDPOINT_NAME)
    print(f"✓ Endpoint '{ENDPOINT_NAME}' already exists")
    print(f"  Status: {endpoint.endpoint_status.state}")
    print(f"  Type: {endpoint.endpoint_type}")
except Exception as e:
    if "not found" in str(e).lower() or "does not exist" in str(e).lower():
        print(f"Creating endpoint '{ENDPOINT_NAME}'...")
        endpoint = w.vector_search_endpoints.create_endpoint(
            name=ENDPOINT_NAME,
            endpoint_type=EndpointType.STANDARD
        )
        print(f"✓ Endpoint creation initiated. This may take a few minutes.")
        
        # Wait for endpoint to be ready
        print("Waiting for endpoint to be online...")
        while True:
            endpoint = w.vector_search_endpoints.get_endpoint(ENDPOINT_NAME)
            status = endpoint.endpoint_status.state.value
            print(f"  Status: {status}")
            if status == "ONLINE":
                print("✓ Endpoint is ready!")
                break
            elif status in ["OFFLINE", "PROVISIONING"]:
                time.sleep(30)
            else:
                print(f"⚠ Unexpected status: {status}")
                break
    else:
        raise

## Step 4: Create Vector Search Index

Create a **Delta Sync index with managed embeddings**. Databricks will:
1. Automatically generate embeddings using `databricks-gte-large-en` (1024 dimensions)
2. Sync automatically when the source table changes (Change Data Feed)
3. Keep embeddings up-to-date as you add/update documents

**Index Configuration:**
- **Index Type**: Delta Sync (auto-syncs from source table)
- **Embedding Model**: `databricks-gte-large-en` (high quality English embeddings)
- **Pipeline Type**: `TRIGGERED` (manual sync on-demand)

In [0]:
# Check if index exists
try:
    existing_index = w.vector_search_indexes.get_index(index_name=INDEX_NAME)
    print(f"✓ Index '{INDEX_NAME}' already exists")
    print(f"  Status: {existing_index.status.detailed_state if hasattr(existing_index.status, 'detailed_state') else str(existing_index.status)}")
    print(f"  Primary Key: {existing_index.primary_key}")
    index_exists = True
except Exception as e:
    if "not found" in str(e).lower() or "does not exist" in str(e).lower():
        print(f"Index '{INDEX_NAME}' does not exist. Creating...")
        index_exists = False
    else:
        raise

# Create index if it doesn't exist
if not index_exists:
    print(f"Creating Delta Sync index with managed embeddings...")
    index = w.vector_search_indexes.create_index(
        name=INDEX_NAME,
        endpoint_name=ENDPOINT_NAME,
        primary_key="doc_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=DOCUMENTS_TABLE,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="content",
                    embedding_model_endpoint_name="databricks-gte-large-en"
                )
            ],
            pipeline_type=PipelineType.TRIGGERED
        )
    )
    print("✓ Index creation initiated")
    
    # Wait for index to be ready
    print("Waiting for index to be ready (this may take several minutes)...")
    for i in range(60):  # Wait up to 30 minutes
        time.sleep(30)
        index_status = w.vector_search_indexes.get_index(index_name=INDEX_NAME)
        status = index_status.status.detailed_state if hasattr(index_status.status, 'detailed_state') else str(index_status.status)
        print(f"  [{i+1}/60] Status: {status}")
        if status == "ONLINE":
            print("✓ Index is ready!")
            break
        elif status in ["PROVISIONING", "SYNCING"]:
            continue
        else:
            print(f"⚠ Status: {status}")
            if index_status.status.message:
                print(f"  Message: {index_status.status.message}")
            break
else:
    print("Skipping index creation - using existing index")

In [0]:
# Check if index is ready before syncing
index_info = w.vector_search_indexes.get_index(index_name=INDEX_NAME)

if not index_info.status.ready:
    print("⚠ Index is not ready yet. Status:")
    print(f"  Ready: {index_info.status.ready}")
    print(f"  Message: {index_info.status.message}")
    print("\nThe index is still provisioning. This typically takes 3-5 minutes.")
    print("Please wait a few minutes, then run this cell again to trigger the sync.")
else:
    # Trigger sync to embed documents
    print("✓ Index is ready. Triggering sync to generate embeddings...")
    w.vector_search_indexes.sync_index(index_name=INDEX_NAME)
    print("✓ Sync triggered. Embeddings are being generated.")
    
    # Check sync status
    time.sleep(5)
    index_info = w.vector_search_indexes.get_index(index_name=INDEX_NAME)
    status_str = index_info.status.detailed_state if hasattr(index_info.status, 'detailed_state') else str(index_info.status)
    print(f"\nIndex Status: {status_str}")
    if index_info.status.message:
        print(f"Message: {index_info.status.message}")

## Step 5: Test Vector Search Retrieval

Test the vector search with sample queries to ensure it returns relevant results. The search performs semantic similarity matching, finding documents with similar meaning (not just keyword matches).

In [0]:
# Test query 1: Find fast charging stations
print("🔍 Query: 'Find fast charging stations near Sydney CBD'\n")

# Check if index is ready before querying
index_info = w.vector_search_indexes.get_index(index_name=INDEX_NAME)

if not index_info.status.ready:
    print("⚠ Index is not ready yet for queries.")
    print(f"  Status: {index_info.status.message}")
    print("\nThe index is still provisioning. This typically takes 3-5 minutes.")
    print("Please wait a few minutes, then run this cell again to test the query.")
else:
    try:
        results = w.vector_search_indexes.query_index(
            index_name=INDEX_NAME,
            columns=["doc_id", "content", "source_table"],
            query_text="Find fast charging stations near Sydney CBD",
            num_results=5
        )
        
        print(f"Found {len(results.result.data_array)} results:\n")
        
        for i, row in enumerate(results.result.data_array, 1):
            doc_id = row[0]
            content = row[1]
            source = row[2]
            score = row[-1]  # Similarity score is last column
            
            print(f"Result {i} (Score: {score:.4f})")
            print(f"Doc ID: {doc_id}")
            print(f"Source: {source}")
            print(f"Content: {content[:300]}...")
            print("-" * 80)
    except Exception as e:
        print(f"⚠ Query failed: {e}")
        print("The index may still be syncing. Wait a few minutes and try again.")

In [0]:
# Test query 2: Route safety information
print("\n🔍 Query: 'What routes have road hazards or safety concerns?'\n")

# Check if index is ready before querying
index_info = w.vector_search_indexes.get_index(index_name=INDEX_NAME)

if not index_info.status.ready:
    print("⚠ Index is not ready yet for queries.")
    print(f"  Status: {index_info.status.message}")
    print("\nThe index is still provisioning. This typically takes 3-5 minutes.")
    print("Please wait a few minutes, then run this cell again to test the query.")
else:
    try:
        results = w.vector_search_indexes.query_index(
            index_name=INDEX_NAME,
            columns=["doc_id", "content", "source_table"],
            query_text="What routes have road hazards or safety concerns?",
            num_results=5
        )
        
        print(f"Found {len(results.result.data_array)} results:\n")
        
        for i, row in enumerate(results.result.data_array, 1):
            doc_id = row[0]
            content = row[1]
            source = row[2]
            score = row[-1]
            
            print(f"Result {i} (Score: {score:.4f})")
            print(f"Doc ID: {doc_id}")
            print(f"Source: {source}")
            print(f"Content: {content[:300]}...")
            print("-" * 80)
    except Exception as e:
        print(f"⚠ Query failed: {e}")
        print("The index may still be syncing. Wait a few minutes and try again.")

## ✅ Vector Search Setup Complete!

You now have:
1. ✓ Documents table with EV charging stations and route safety data
2. ✓ Vector search endpoint (`nsw-ev-rag-endpoint`)
3. ✓ Delta Sync index with automatic embeddings (`mobility_ai.rag.ev_documents_index`)
4. ✓ Working semantic search retrieval

## Next Steps for RAG Application

### 1. **Build RAG Pipeline**
   - Create retrieval function that queries the vector index
   - Integrate with LLM (DBRX, Llama, GPT-4) for generation
   - Build prompt template that combines retrieved context with user query

### 2. **Extend Your Databricks App**
   - Add chat interface (Gradio/Streamlit)
   - Implement RAG workflow: Query → Retrieve → Generate
   - Deploy as interactive web app

### 3. **Enhance Documents**
   - Add more document types (weather impacts, charging network policies)
   - Include regional infrastructure summaries
   - Create FAQ documents from common user questions

### 4. **Production Optimizations**
   - Set pipeline_type to CONTINUOUS for auto-sync
   - Add filters for location-based retrieval
   - Monitor query performance and costs
   - Implement caching for common queries

## Useful Commands

```python
# Manually sync index when documents change
w.vector_search_indexes.sync_index(index_name=INDEX_NAME)

# Check index status
index_info = w.vector_search_indexes.get_index(index_name=INDEX_NAME)
print(index_info.status)

# Query with filters (for standard endpoint)
results = w.vector_search_indexes.query_index(
    index_name=INDEX_NAME,
    query_text="your query",
    num_results=10,
    filters_json='{"source_table": "mobility_ai.silver.ev_charging_stations"}'
)
```